In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import math
import torch.nn.functional as F
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

In [ ]:
MASTER_CSV = "out/master_dataset.csv"
OUTPUT_FILE = "embeddings/video_sequences_v1.pt"
ORIGINAL_FPS = 30
TARGET_FPS = 5
NUM_WORKERS = 4  # 8-CPU allocation!

def process_single_video(args):
    """
    Worker function that processes ONE video file, extracting ALL utterances 
    from it at the same time to drastically reduce I/O read times.
    """
    npz_path, group_df, max_video_len = args
    results = {}
    au_feature_count = 0
    
    try:
        # 1. LOAD THE FILE EXACTLY ONCE
        d = np.load(npz_path, allow_pickle=True)
        
        # --- SAFE GRAB: FAUs ---
        if 'movement_v4:FAUValue' in d.files: au_matrix = d['movement_v4:FAUValue']
        elif 'movement:FAUValue' in d.files: au_matrix = d['movement:FAUValue']
        else: return {}, 0 # If there are no faces, we can't do anything!
        
        num_frames = au_matrix.shape[0]

        # --- SAFE GRAB: Head & Gaze ---
        if 'movement_v4:head_encodings' in d.files: head_matrix = d['movement_v4:head_encodings']
        elif 'movement:head_encodings' in d.files: head_matrix = d['movement:head_encodings']
        else: head_matrix = np.zeros((num_frames, 6))

        if 'movement_v4:gaze_encodings' in d.files: gaze_matrix = d['movement_v4:gaze_encodings']
        elif 'movement:gaze_encodings' in d.files: gaze_matrix = d['movement:gaze_encodings']
        else: gaze_matrix = np.zeros((num_frames, 6))

        # --- SAFE GRAB: Body Pose ---
        if 'smplh:body_pose' in d.files: body_matrix = d['smplh:body_pose']
        else: body_matrix = np.zeros((num_frames, 63))

        # ==========================================
        # THE FIX: Flatten all matrices to 2D!
        # This converts [Frames, Joints, 3] -> [Frames, Joints * 3]
        # ==========================================
        au_matrix = au_matrix.reshape(au_matrix.shape[0], -1)
        head_matrix = head_matrix.reshape(head_matrix.shape[0], -1)
        gaze_matrix = gaze_matrix.reshape(gaze_matrix.shape[0], -1)
        body_matrix = body_matrix.reshape(body_matrix.shape[0], -1)

        # Ensure everything has the same number of frames
        min_frames = min(au_matrix.shape[0], head_matrix.shape[0], gaze_matrix.shape[0], body_matrix.shape[0])
        
        # Stack them horizontally!
        combined_matrix = np.column_stack([
            au_matrix[:min_frames], 
            head_matrix[:min_frames], 
            gaze_matrix[:min_frames], 
            body_matrix[:min_frames]
        ])
            
        au_feature_count = combined_matrix.shape[1]
        
        # 2. SLICE ALL UTTERANCES FROM THIS VIDEO
        for _, row in group_df.iterrows():
            sample_id = row['sample_id']
            start_sec = row['start_sec']
            end_sec = row['end_sec']
            
            start_frame = int(start_sec * ORIGINAL_FPS)
            end_frame = int(end_sec * ORIGINAL_FPS)
            
            end_frame = min(end_frame, combined_matrix.shape[0])
            utterance_aus = combined_matrix[start_frame:end_frame]

            # Downsample to TARGET_FPS
            step_size = ORIGINAL_FPS // TARGET_FPS
            if step_size > 0 and len(utterance_aus) > step_size:
                downsampled_aus = utterance_aus[::step_size]
            else:
                downsampled_aus = utterance_aus

            seq_tensor = torch.tensor(downsampled_aus, dtype=torch.float32)

            # Dynamic Padding/Truncation
            current_len = seq_tensor.shape[0]
            if current_len > max_video_len:
                seq_tensor = seq_tensor[:max_video_len, :]
            elif current_len < max_video_len:
                pad_amount = max_video_len - current_len
                seq_tensor = F.pad(seq_tensor, (0, 0, 0, pad_amount), "constant", 0)

            results[sample_id] = seq_tensor
            
        return results, au_feature_count

    except Exception as e:
        print(f"\nFailed on {npz_path}: {e}")
        return {}, 0


def extract_video_sequences():
    df = pd.read_csv(MASTER_CSV)

    # --- PADDING ---
    p95_duration = df['duration_sec'].quantile(0.95)
    MAX_VIDEO_LEN = math.ceil(p95_duration * TARGET_FPS)
    print(f"95th Percentile Duration: {p95_duration:.2f}s")
    print(f"MAX_VIDEO_LEN set to: {MAX_VIDEO_LEN} frames")

    # --- CHECKPOINT RESUME ---
    if os.path.exists(OUTPUT_FILE):
        print(f"Loading existing progress from {OUTPUT_FILE}...")
        video_dict = torch.load(OUTPUT_FILE)
        processed_ids = set(video_dict.keys())
        df_to_process = df[~df['sample_id'].isin(processed_ids)]
        print(f"Resuming! {len(processed_ids)} already extracted, {len(df_to_process)} remaining.")
    else:
        video_dict = {}
        df_to_process = df
        
    if len(df_to_process) == 0:
        print("All video features are already extracted!")
        return

    # Group the dataframe so we pass all utterances belonging to the same NPZ file together
    grouped_tasks = []
    for npz_path, group in df_to_process.groupby('npz_path'):
        grouped_tasks.append((npz_path, group, MAX_VIDEO_LEN))

    print(f"\nGrouped into {len(grouped_tasks)} unique video files.")
    print(f"Starting Multi-Core Extraction ({NUM_WORKERS} Workers)...")
    
    global_feature_count = 0

    # --- MULTIPROCESSING ---
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {executor.submit(process_single_video, task): task for task in grouped_tasks}
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing NPZ Files"):
            try:
                batch_results, feat_count = future.result()
                if batch_results:
                    video_dict.update(batch_results)
                    global_feature_count = max(global_feature_count, feat_count)
            except Exception as e:
                pass
                
    # Final save when complete
    torch.save(video_dict, OUTPUT_FILE)
    print(f"\nSUCCESS! Saved Video Sequences to {OUTPUT_FILE} (Shape: {MAX_VIDEO_LEN}, {global_feature_count})")

if __name__ == "__main__":
    extract_video_sequences()

95th Percentile Duration: 12.93s
Data-Driven MAX_VIDEO_LEN set to: 65 frames

Grouped into 534 unique video files.
Starting Multi-Core Extraction (4 Workers)...


Processing NPZ Files: 100%|██████████| 534/534 [00:05<00:00, 104.46it/s]



✅ SUCCESS! Saved Video Sequences to temp/video_sequences.pt (Shape: 65, 92)
